# 02 – Silver Layer: EMS Incident Dispatch Cleaning

**Purpose:** Turn raw Bronze data into clean, typed, trustworthy data.

**Input:** Latest snapshot in `bronze/ems_incident_dispatch/`
**Output:** Silver tables used for the Gold analysis (response time by borough and time period, and by call type and severity).

**Silver rules:** fix types, handle missing values, standardize categories, remove invalid or duplicate rows, and document every rule.

## 1. Load configuration

Paths and settings are stored in `config/dev.yaml` and `config/prod.yaml`, not in the code. Change `ENV` to `"prod"` to run on the full dataset.

In [4]:
import sys
sys.path.append("../src")          # lets the notebook import files from src/

from config_loader import load_config

ENV = "prod"                        # change to "prod" for the full data
config = load_config(f"../config/{ENV}.yaml")

print(config)

{'environment': 'prod', 'spark': {'driver_memory': '8g', 'cores': 4, 'shuffle_partitions': 200}, 'source': {'dataset_name': 'ems_incident_dispatch', 'dataset_id': '76xm-jjuj', 'input_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv'}, 'bronze': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/bronze'}, 'silver': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/silver'}}


## 2. Start Spark

Create the Spark session, which is the entry point for all Spark operations. `getOrCreate()` reuses an existing session if one is already running. The printed version confirms Spark is working.

In [5]:
from pyspark.sql import SparkSession

spark_cfg = config["spark"]

spark = (
    SparkSession.builder
    .appName("ems_pipeline")
    .master(f"local[{spark_cfg['cores']}]")
    .config("spark.driver.memory", spark_cfg["driver_memory"])
    .config("spark.sql.shuffle.partitions", spark_cfg["shuffle_partitions"])
    .getOrCreate()
)

print(spark.version)
print("Master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 11:01:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.0
Master: local[4]
Driver memory: 8g


## 3. Read the latest Bronze snapshot

Bronze snapshots are stored in folders named `snapshot_date=YYYY-MM-DD`. Sorting the folder names gives us the newest one. We read it as Parquet, so all columns are still raw strings.

In [6]:
import os, glob

bronze_root = os.path.abspath(os.path.join(
    "..", config["bronze"]["root_path"], config["source"]["dataset_name"]
))

snapshots = sorted(glob.glob(os.path.join(bronze_root, "snapshot_date=*")))
print("Snapshots found:", [os.path.basename(s) for s in snapshots])

latest_snapshot = snapshots[-1]
print("Using:", latest_snapshot)

bronze_df = spark.read.parquet(latest_snapshot)
total_rows = bronze_df.count()
print("Rows:", total_rows)

Snapshots found: ['snapshot_date=2026-11-02']
Using: /home/ubuntu/spark-notebooks/ems-pipeline/data/bronze/ems_incident_dispatch/snapshot_date=2026-11-02


Rows: 30389767


## 4. Profile missing values

In the raw data, missing values may appear in three different ways: a real null, the text `NULL`, or an empty string. This table counts all three for every source column, so we can decide how to handle each one.

In [7]:
import pandas as pd
from pyspark.sql import functions as F

cols = [c for c in bronze_df.columns if not c.startswith("_")]

aggs = []
for c in cols:
    aggs.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}__real_null"))
    aggs.append(F.sum(F.when(F.trim(F.col(c)).isin("NULL", ""), 1).otherwise(0)).alias(f"{c}__null_text"))

counts = bronze_df.agg(*aggs).collect()[0].asDict()

profile = pd.DataFrame({
    "column": cols,
    "real_null": [counts[f"{c}__real_null"] for c in cols],
    "null_text_or_empty": [counts[f"{c}__null_text"] for c in cols],
})
profile["missing_total"] = profile["real_null"] + profile["null_text_or_empty"]
profile["missing_pct"] = (100 * profile["missing_total"] / total_rows).round(1)

pd.set_option("display.max_rows", 100)
profile.sort_values("missing_total", ascending=False).reset_index(drop=True)

26/11/09 11:02:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,column,real_null,null_text_or_empty,missing_total,missing_pct
0,FIRST_HOSP_ARRIVAL_DATETIME,9971312,0,9971312,32.8
1,FIRST_TO_HOSP_DATETIME,9799315,0,9799315,32.2
2,INCIDENT_RESPONSE_SECONDS_QY,1128836,0,1128836,3.7
3,INCIDENT_TRAVEL_TM_SECONDS_QY,1123304,0,1123304,3.7
4,FIRST_ON_SCENE_DATETIME,1122848,0,1122848,3.7
5,COMMUNITYSCHOOLDISTRICT,614342,0,614342,2.0
6,ZIPCODE,601029,0,601029,2.0
7,COMMUNITYDISTRICT,600069,0,600069,2.0
8,POLICEPRECINCT,599974,0,599974,2.0
9,CITYCOUNCILDISTRICT,599974,0,599974,2.0


## Silver transformations

The cleaning rules live in `src/silver_rules.py`, where they are covered by automated tests. This notebook calls them in order: convert types, add flags and derived fields, reject unusable rows, deduplicate by incident ID, and flag logical problems.

In [8]:
from silver_rules import (
    convert_types, add_flags_and_derived, split_rejected,
    dedupe_by_incident_id, flag_logical_issues, logical_check_expressions,
)

typed_df = convert_types(bronze_df)
silver_df = add_flags_and_derived(typed_df)
accepted_df, rejected_df = split_rejected(silver_df)
deduped_df = dedupe_by_incident_id(accepted_df)
clean_df = flag_logical_issues(deduped_df)

checks = logical_check_expressions()   # used by the check-count cell

## 5. Convert data types (correct formatting issues)

Bronze stores everything as text. Silver converts each column to its proper type:

- **Date-time columns** such as `01/01/2005 12:00:24 AM` become real timestamps.
- **Numeric columns** such as `942.0` or `5` become integers.
- **ZIP code** stays text (it is an identifier, not a number), but loses the trailing `.0` and keeps 5 digits.

A value that cannot be converted becomes null, which would silently hide a problem. So we also count conversion failures: values that existed before conversion but are null after it. We expect zero.

In [9]:
from pyspark.sql import functions as F

TS_FORMAT = "MM/dd/yyyy hh:mm:ss a"

timestamp_cols = [
    "INCIDENT_DATETIME", "FIRST_ASSIGNMENT_DATETIME", "FIRST_ACTIVATION_DATETIME",
    "FIRST_ON_SCENE_DATETIME", "FIRST_TO_HOSP_DATETIME", "FIRST_HOSP_ARRIVAL_DATETIME",
    "INCIDENT_CLOSE_DATETIME",
]

int_cols = [
    "INITIAL_SEVERITY_LEVEL_CODE", "FINAL_SEVERITY_LEVEL_CODE",
    "DISPATCH_RESPONSE_SECONDS_QY", "INCIDENT_RESPONSE_SECONDS_QY",
    "INCIDENT_TRAVEL_TM_SECONDS_QY", "POLICEPRECINCT", "CITYCOUNCILDISTRICT",
    "COMMUNITYDISTRICT", "COMMUNITYSCHOOLDISTRICT", "CONGRESSIONALDISTRICT",
]

# Each conversion is defined once, then used for both the check and the real conversion
conversions = {c: F.to_timestamp(F.col(c), TS_FORMAT) for c in timestamp_cols}
conversions.update({c: F.col(c).cast("double").cast("int") for c in int_cols})

# Conversion failures: value existed before, but is null after conversion
fail_aggs = [
    F.sum(F.when(F.col(c).isNotNull() & expr.isNull(), 1).otherwise(0)).alias(c)
    for c, expr in conversions.items()
]
failures = bronze_df.agg(*fail_aggs).collect()[0].asDict()

print("Conversion failures per column:")
for c, n in failures.items():
    print(f"  {c}: {n}")

# Apply the conversions
typed_df = bronze_df.withColumns(conversions)
typed_df = typed_df.withColumn(
    "ZIPCODE",
    F.lpad(F.col("ZIPCODE").cast("double").cast("int").cast("string"), 5, "0"),
)

typed_df.select(timestamp_cols[:2] + ["INCIDENT_RESPONSE_SECONDS_QY", "ZIPCODE"]).printSchema()
typed_df.select("INCIDENT_ID", "INCIDENT_DATETIME", "INCIDENT_RESPONSE_SECONDS_QY", "ZIPCODE").show(5, truncate=False)

Conversion failures per column:
  INCIDENT_DATETIME: 0
  FIRST_ASSIGNMENT_DATETIME: 0
  FIRST_ACTIVATION_DATETIME: 0
  FIRST_ON_SCENE_DATETIME: 0
  FIRST_TO_HOSP_DATETIME: 0
  FIRST_HOSP_ARRIVAL_DATETIME: 0
  INCIDENT_CLOSE_DATETIME: 0
  INITIAL_SEVERITY_LEVEL_CODE: 0
  FINAL_SEVERITY_LEVEL_CODE: 0
  DISPATCH_RESPONSE_SECONDS_QY: 0
  INCIDENT_RESPONSE_SECONDS_QY: 0
  INCIDENT_TRAVEL_TM_SECONDS_QY: 0
  POLICEPRECINCT: 0
  CITYCOUNCILDISTRICT: 0
  COMMUNITYDISTRICT: 0
  COMMUNITYSCHOOLDISTRICT: 0
  CONGRESSIONALDISTRICT: 0
root
 |-- INCIDENT_DATETIME: timestamp (nullable = true)
 |-- FIRST_ASSIGNMENT_DATETIME: timestamp (nullable = true)
 |-- INCIDENT_RESPONSE_SECONDS_QY: integer (nullable = true)
 |-- ZIPCODE: string (nullable = true)

+-----------+-------------------+----------------------------+-------+
|INCIDENT_ID|INCIDENT_DATETIME  |INCIDENT_RESPONSE_SECONDS_QY|ZIPCODE|
+-----------+-------------------+----------------------------+-------+
|83533484   |2026-04-07 12:40:07|3888     

## 6. Add validity flags and derived fields

**Flags (decision: flag, don't reject):** The source columns `VALID_DISPATCH_RSPNS_TIME_INDC` and `VALID_INCIDENT_RSPNS_TIME_INDC` contain `Y`/`N`. We convert them to true/false columns, so records are kept but can be filtered in Gold. A missing flag is treated as `false`, the cautious choice.

**Derived fields:** date, year, hour, and day of week from the incident timestamp, for the time-period analysis in Gold.

**Standardized borough:** `RICHMOND / STATEN ISLAND` becomes `STATEN ISLAND`. The original `BOROUGH` column is kept untouched, so nothing is lost.

**Consistency check:** a record flagged valid should have response seconds, and a record flagged invalid should not. Any mismatch would show that the flag cannot be trusted.

In [10]:
silver_df = (
    typed_df
    .withColumn("is_valid_dispatch_response",
                F.coalesce(F.col("VALID_DISPATCH_RSPNS_TIME_INDC") == "Y", F.lit(False)))
    .withColumn("is_valid_incident_response",
                F.coalesce(F.col("VALID_INCIDENT_RSPNS_TIME_INDC") == "Y", F.lit(False)))
    .withColumn("incident_date", F.to_date("INCIDENT_DATETIME"))
    .withColumn("incident_year", F.year("INCIDENT_DATETIME"))
    .withColumn("incident_hour", F.hour("INCIDENT_DATETIME"))
    .withColumn("incident_day_of_week", F.date_format("INCIDENT_DATETIME", "EEEE"))
    .withColumn("borough_std",
                F.when(F.col("BOROUGH") == "RICHMOND / STATEN ISLAND", "STATEN ISLAND")
                 .otherwise(F.col("BOROUGH")))
)

print("Flag consistency check (incident response):")
(silver_df.groupBy("is_valid_incident_response")
    .agg(F.count("*").alias("rows"),
         F.sum(F.when(F.col("INCIDENT_RESPONSE_SECONDS_QY").isNull(), 1).otherwise(0))
          .alias("rows_with_null_response_seconds"))
    .show())

silver_df.groupBy("borough_std").count().orderBy(F.desc("count")).show()

silver_df.select("INCIDENT_ID", "incident_date", "incident_year", "incident_hour",
                 "incident_day_of_week", "borough_std",
                 "is_valid_dispatch_response", "is_valid_incident_response").show(5, truncate=False)

Flag consistency check (incident response):


+--------------------------+--------+-------------------------------+
|is_valid_incident_response|    rows|rows_with_null_response_seconds|
+--------------------------+--------+-------------------------------+
|                      true|29260931|                              0|
|                     false| 1128836|                        1128836|
+--------------------------+--------+-------------------------------+



+-------------+-------+
|  borough_std|  count|
+-------------+-------+
|     BROOKLYN|8614588|
|    MANHATTAN|7471621|
|        BRONX|7068231|
|       QUEENS|5967030|
|STATEN ISLAND|1268078|
|      UNKNOWN|    219|
+-------------+-------+

+-----------+-------------+-------------+-------------+--------------------+-----------+--------------------------+--------------------------+
|INCIDENT_ID|incident_date|incident_year|incident_hour|incident_day_of_week|borough_std|is_valid_dispatch_response|is_valid_incident_response|
+-----------+-------------+-------------+-------------+--------------------+-----------+--------------------------+--------------------------+
|83533484   |2026-04-07   |2026         |12           |Tuesday             |MANHATTAN  |true                      |true                      |
|83533485   |2026-04-07   |2026         |12           |Tuesday             |MANHATTAN  |true                      |true                      |
|83533486   |2026-04-07   |2026         |12 

## 7. Reject, deduplicate, and check logic

**Reject (can't be used at all):** records with no `INCIDENT_ID` or no `INCIDENT_DATETIME`. They are kept in a separate `rejected_df` so they stay visible and countable.

**Deduplicate:** exact duplicate records (all source columns identical) are removed.

**Logical checks (flag, don't reject):** events in an emergency should happen in order. A record is flagged `has_data_issue = true` if:
- assignment happens before the incident was created
- the unit arrives on scene before it was assigned
- hospital arrival happens before departure for the hospital
- the incident closes before it was created
- any response-time value is negative

Missing timestamps are not errors (not every incident has a hospital trip), so a check only applies when both timestamps exist.

In [11]:
from functools import reduce

source_cols = [c for c in bronze_df.columns if not c.startswith("_")]
rows_in = bronze_df.count()

# 1. Reject rows with no key or no incident time
has_key = F.col("INCIDENT_ID").isNotNull() & F.col("INCIDENT_DATETIME").isNotNull()
rejected_df = silver_df.filter(~has_key)
accepted_df = silver_df.filter(has_key)

# 2. Remove exact duplicates (compare source columns only, not metadata)
deduped_df = accepted_df.dropDuplicates(source_cols)

# 3. Logical checks
def out_of_order(earlier, later):
    return (F.col(earlier).isNotNull() & F.col(later).isNotNull()
            & (F.col(later) < F.col(earlier)))

def is_negative(col):
    return F.coalesce(F.col(col) < 0, F.lit(False))

checks = {
    "assignment_before_incident": out_of_order("INCIDENT_DATETIME", "FIRST_ASSIGNMENT_DATETIME"),
    "on_scene_before_assignment": out_of_order("FIRST_ASSIGNMENT_DATETIME", "FIRST_ON_SCENE_DATETIME"),
    "hospital_arrival_before_departure": out_of_order("FIRST_TO_HOSP_DATETIME", "FIRST_HOSP_ARRIVAL_DATETIME"),
    "close_before_incident": out_of_order("INCIDENT_DATETIME", "INCIDENT_CLOSE_DATETIME"),
    "negative_response_seconds": (is_negative("DISPATCH_RESPONSE_SECONDS_QY")
                                  | is_negative("INCIDENT_RESPONSE_SECONDS_QY")
                                  | is_negative("INCIDENT_TRAVEL_TM_SECONDS_QY")),
}

check_counts = deduped_df.agg(
    *[F.sum(F.when(expr, 1).otherwise(0)).alias(name) for name, expr in checks.items()]
).collect()[0].asDict()

print("Records failing each logical check:")
for name, n in check_counts.items():
    print(f"  {name}: {n}")

# One combined flag
any_issue = reduce(lambda a, b: a | b, checks.values())
clean_df = deduped_df.withColumn("has_data_issue", any_issue)

Records failing each logical check:
  assignment_before_incident: 176
  on_scene_before_assignment: 450
  hospital_arrival_before_departure: 13741
  close_before_incident: 1140
  negative_response_seconds: 0


## 8. Quality summary (before and after)

This table counts what entered Silver, what was rejected or removed, what was flagged, and what remains. The check at the end proves that no row has been lost without being accounted for.

In [12]:
import pandas as pd

rows_rejected = rejected_df.count()
rows_out = clean_df.count()
rows_dupes_removed = accepted_df.count() - rows_out

flag_counts = clean_df.agg(
    F.sum(F.when(~F.col("is_valid_dispatch_response"), 1).otherwise(0)).alias("invalid_dispatch"),
    F.sum(F.when(~F.col("is_valid_incident_response"), 1).otherwise(0)).alias("invalid_incident"),
    F.sum(F.when(F.col("has_data_issue"), 1).otherwise(0)).alias("data_issue"),
).collect()[0].asDict()

quality_summary = pd.DataFrame([
    ("Rows in (Bronze)", rows_in),
    ("Rejected (no ID or no incident time)", rows_rejected),
    ("Exact duplicates removed", rows_dupes_removed),
    ("Rows out (Silver)", rows_out),
    ("Flagged: invalid dispatch response", flag_counts["invalid_dispatch"]),
    ("Flagged: invalid incident response", flag_counts["invalid_incident"]),
    ("Flagged: logical data issue", flag_counts["data_issue"]),
], columns=["metric", "count"])

assert rows_in == rows_rejected + rows_dupes_removed + rows_out, "Row counts do not add up!"
quality_summary

,metric,count
0,Rows in (Bronze),30389767
1,Rejected (no ID or no incident time),0
2,Exact duplicates removed,0
3,Rows out (Silver),30389767
4,Flagged: invalid dispatch response,242125
5,Flagged: invalid incident response,1128836
6,Flagged: logical data issue,15290


## 9. Write the Silver outputs

**Silver table:** saved as Parquet, partitioned by `incident_year`. Partitioning splits the data into one folder per year, so later queries that filter on a year read only those folders. Across about 20 years, this gives manageable partitions of around a million rows each.

**Overwrite is correct here.** Unlike Bronze, Silver is rebuilt from a Bronze snapshot, so re-running replaces the old Silver table with the same result (this is called idempotent).

**Lineage columns:** `_bronze_snapshot_date` and `_silver_processed_at` record which Bronze snapshot this Silver data came from and when.

**Rejected records:** saved separately with a reason, so they stay visible and countable. The folder is empty of rows if nothing was rejected.

In [13]:
silver_root = os.path.abspath(os.path.join(
    "..", config["silver"]["root_path"], config["source"]["dataset_name"]
))
silver_data_path = os.path.join(silver_root, "data")
rejected_path = os.path.join(silver_root, "rejected")

bronze_snapshot_date = os.path.basename(latest_snapshot).split("=")[1]

silver_out = (
    clean_df
    .withColumn("_bronze_snapshot_date", F.lit(bronze_snapshot_date))
    .withColumn("_silver_processed_at", F.current_timestamp())
)

rejected_out = rejected_df.withColumn(
    "_reject_reason", F.lit("missing incident id or incident time")
)

silver_out.write.mode("overwrite").partitionBy("incident_year").parquet(silver_data_path)
rejected_out.write.mode("overwrite").parquet(rejected_path)

print("Silver table written to:", silver_data_path)
print("Rejected records written to:", rejected_path)

Silver table written to: /home/ubuntu/spark-notebooks/ems-pipeline/data/silver/ems_incident_dispatch/data
Rejected records written to: /home/ubuntu/spark-notebooks/ems-pipeline/data/silver/ems_incident_dispatch/rejected


## 10. Save the quality summary

The before/after counts are saved as a JSON file, just like the Bronze run log, so every Silver run leaves auditable evidence of what entered, what was rejected, and what was flagged.

In [14]:
import json
from datetime import datetime, timezone

quality_dir = os.path.join(silver_root, "_quality_log")
os.makedirs(quality_dir, exist_ok=True)

quality_record = {
    "environment": config["environment"],
    "bronze_snapshot_date": bronze_snapshot_date,
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": {row.metric: int(row["count"]) for _, row in quality_summary.iterrows()},
}

quality_file = os.path.join(quality_dir, f"silver_quality_{bronze_snapshot_date}.json")
with open(quality_file, "w") as f:
    json.dump(quality_record, f, indent=2)

print(json.dumps(quality_record, indent=2))

{
  "environment": "prod",
  "bronze_snapshot_date": "2026-11-02",
  "processed_at_utc": "2026-11-09T11:17:23.930621+00:00",
  "metrics": {
    "Rows in (Bronze)": 30389767,
    "Rejected (no ID or no incident time)": 0,
    "Exact duplicates removed": 0,
    "Rows out (Silver)": 30389767,
    "Flagged: invalid dispatch response": 242125,
    "Flagged: invalid incident response": 1128836,
    "Flagged: logical data issue": 15290
  }
}


### Verify the Silver output

Read the Silver table back from disk. The row count must equal "Rows out (Silver)" from the quality summary, the types must still be correct, and the year partition folders must exist.

In [15]:
check_silver = spark.read.parquet(silver_data_path)

print("Silver rows on disk:", check_silver.count())
print("Rows out per quality summary:", rows_out)
print("Rejected rows on disk:", spark.read.parquet(rejected_path).count())
print("Partition folders:", sorted(d for d in os.listdir(silver_data_path) if d.startswith("incident_year=")))

check_silver.select("INCIDENT_ID", "INCIDENT_DATETIME", "incident_year", "borough_std",
                    "is_valid_incident_response", "has_data_issue").printSchema()

Silver rows on disk: 30389767
Rows out per quality summary: 30389767
Rejected rows on disk: 0
Partition folders: ['incident_year=2005', 'incident_year=2006', 'incident_year=2007', 'incident_year=2008', 'incident_year=2009', 'incident_year=2010', 'incident_year=2011', 'incident_year=2012', 'incident_year=2013', 'incident_year=2014', 'incident_year=2015', 'incident_year=2016', 'incident_year=2017', 'incident_year=2018', 'incident_year=2019', 'incident_year=2020', 'incident_year=2021', 'incident_year=2022', 'incident_year=2023', 'incident_year=2024', 'incident_year=2025', 'incident_year=2026']
root
 |-- INCIDENT_ID: string (nullable = true)
 |-- INCIDENT_DATETIME: timestamp (nullable = true)
 |-- incident_year: integer (nullable = true)
 |-- borough_std: string (nullable = true)
 |-- is_valid_incident_response: boolean (nullable = true)
 |-- has_data_issue: boolean (nullable = true)

